
# 📊 COCO Object Detection Evaluation Dashboard
This notebook allows you to evaluate object detection results against the COCO dataset.

### 🛠️ Setup
We will install `pycocotools` for the evaluation metrics and fetch ground truth data from Official website.

**You do not need to connect a GPU runtime, CPU env is fine.**

In [ ]:
!pip install -q pycocotools datasets gdown

## Download COCO Annotations
This version automatically pulls the **COCO 2017 Validation set** metadata from Official COCO website. Use this if your model was tested on the standard COCO benchmark.

In [ ]:
import os
import json
import requests
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# 2. Download the OFFICIAL COCO 2017 Val Annotations (Direct from source)
# This is a large file (~19MB), but it is the industry standard.
print("Downloading official COCO 2017 validation annotations...")
!wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -q annotations_trainval2017.zip

# 3. Initialize Ground Truth
ann_file = 'annotations/instances_val2017.json'
coco_gt = COCO(ann_file)
print(coco_gt)

## Upload Predictions
This step uses a sample predictions file for RetinaNet model [predictions](https://storage.googleapis.com/predictions_colab/predictions.json.zip), and compares it against the ground truth loaded above.
You can also upload your `predictions.json' file.

In [ ]:
# Version 1: Use given predictions.json file

import zipfile
from pathlib import Path
import os

!wget -O /content/predictions.zip https://storage.googleapis.com/predictions_colab/predictions.json.zip

ZIP_PATH = Path("/content/predictions.zip")   # TODO: change if your uploaded file has a different name
PRED_DIR = Path("/content/")

assert ZIP_PATH.exists(), f"Zip file not found: {ZIP_PATH}"


with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(PRED_DIR)

print("Extracted to:", PRED_DIR)
predictions_file_name = "predictions.json"


**Alternatively, if you have a predictions file, here you can upload it.**

In [ ]:
# # Version 2. Upload your predictions.json file

# from google.colab import files

# print("Please upload your predictions JSON file:")
# uploaded = files.upload()

# # 2. Get the filename from the upload dictionary
# if uploaded:
#     # Get the name of the first file uploaded
#     uploaded_filename = list(uploaded.keys())[0]
#     print(f"✅ Received: {uploaded_filename}")
# predictions_file_name = uploaded_filename


##  Run Metrics
Using official API.

In [ ]:
# 2. Load and Evaluate
coco_dt = coco_gt.loadRes(predictions_file_name)
coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()